# Tugas 2 - Text Preprocessing, TF-IDF, ICSDF, dan PCA

Pada tugas ini dilakukan pengolahan terhadap 200 berita dari Detik.com yang terdiri dari 100 berita kategori Sport dan 100 berita kategori Finance.

Tahapan yang dilakukan meliputi eksplorasi data, pemeriksaan kualitas teks, normalisasi karakter, tokenisasi, normalisasi kata tidak baku dan istilah asing, pembentukan representasi TF-IDF, pembobotan ICSDF, seleksi fitur, serta reduksi dimensi menggunakan PCA.

Selain itu dilakukan analisis kata unik berdasarkan kategori, analisis kata dengan bobot TF-IDF tertinggi, visualisasi PCA dua dimensi, dan analisis PCA Feature Loading.

## 2. Import Library

Setelah library tersedia, library yang diperlukan diimpor ke dalam notebook.

`pandas` digunakan untuk membaca dan mengolah dataset. `numpy` digunakan untuk perhitungan numerik. `re` digunakan untuk preprocessing teks dengan regular expression. Library dari `scikit-learn` digunakan untuk pembagian data, TF-IDF, dan PCA.

In [3]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

## 3. Membaca Dataset

Dataset yang digunakan merupakan kumpulan 200 berita yang terdiri dari dua kategori, yaitu Sport dan Finance.

Dataset memiliki tiga kolom utama, yaitu:

- `id` sebagai identitas berita
- `isi_berita` sebagai isi teks berita
- `label` sebagai kategori berita

Masing-masing kategori terdiri dari 100 berita.

In [4]:
df = pd.read_excel("dataset_detik_200_berita.xlsx")

print("Jumlah data :", len(df))
print("\nNama kolom :")
print(df.columns)

print("\nJumlah data per label :")
print(df["label"].value_counts())

Jumlah data : 200

Nama kolom :
Index(['id', 'isi_berita', 'label'], dtype='str')

Jumlah data per label :
label
sport      100
finance    100
Name: count, dtype: int64


## 4. Mengubah Label Menjadi Numerik

Label teks perlu diubah menjadi bentuk numerik agar dapat digunakan pada proses pengolahan data selanjutnya.

Pada tugas ini digunakan aturan:

- `sport` menjadi `1`
- `finance` menjadi `0`

Dengan demikian setiap berita mempunyai label numerik yang mewakili kategorinya.

In [6]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

df[["id", "label", "label_num"]].head()

,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


## 5. Menghitung Jumlah Kata Semua Berita

Sebelum dilakukan preprocessing, jumlah kata pada setiap berita dihitung terlebih dahulu.

Perhitungan dilakukan dengan memisahkan teks berdasarkan spasi menggunakan fungsi `split()`.

Hasil perhitungan disimpan pada kolom `jumlah_kata_asli`.

In [7]:
df["jumlah_kata_asli"] = (
    df["isi_berita"]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

print("Total seluruh kata :", df["jumlah_kata_asli"].sum())

df[["id", "label", "jumlah_kata_asli"]].head()

Total seluruh kata : 67076


,id,label,jumlah_kata_asli
0,1,sport,320
1,2,sport,323
2,3,sport,256
3,4,sport,238
4,5,sport,402


## 6. Kamus Kata Tidak Baku

Kamus kata tidak baku digunakan untuk mengubah kata singkatan atau kata tidak baku menjadi bentuk yang lebih standar.

Contohnya adalah:

- `yg` menjadi `yang`
- `dgn` menjadi `dengan`
- `utk` menjadi `untuk`
- `krn` menjadi `karena`
- `tdk` menjadi `tidak`
- `dr` menjadi `dari`

Normalisasi ini dilakukan agar kata yang mempunyai makna sama tidak dianggap sebagai kata yang berbeda.

In [8]:
kamus_tidak_baku = {
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "kalo": "kalau",
    "kalok": "kalau",
    "aja": "saja",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "tdk": "tidak",
    "dr": "dari",
    "dlm": "dalam",
    "jd": "jadi",
    "bgt": "banget",
    "tp": "tetapi",
    "tapi": "tetapi",
    "karna": "karena",
    "trus": "terus",
    "kmrn": "kemarin",
    "dpt": "dapat",
    "hrs": "harus",
    "sm": "sama",
    "sy": "saya"
}

## 7. Kamus Bahasa Asing

Kamus bahasa asing digunakan untuk menormalisasi beberapa istilah asing yang terdapat pada berita menjadi istilah bahasa Indonesia.

Kamus disesuaikan dengan dua kategori dataset, yaitu Sport dan Finance.

Contohnya pada kategori Sport:

- `rider` menjadi `pembalap`
- `race` menjadi `balapan`
- `team` menjadi `tim`
- `player` menjadi `pemain`

Sedangkan pada kategori Finance:

- `finance` menjadi `keuangan`
- `market` menjadi `pasar`
- `stock` menjadi `saham`
- `investment` menjadi `investasi`

In [19]:
kamus_asing = {
    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",

    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}

print("Kamus bahasa asing berhasil dibuat.")

Kamus bahasa asing berhasil dibuat.


## 8. Fungsi Preprocessing

Preprocessing dilakukan untuk membersihkan teks sebelum digunakan dalam proses pembobotan.

Tahapan preprocessing yang dilakukan adalah:

1. Mengubah teks menjadi string.
2. Case folding dengan mengubah seluruh huruf menjadi huruf kecil.
3. Menghapus URL.
4. Menghapus alamat email.
5. Menghapus angka.
6. Menghapus tanda baca, simbol, dan emotikon.
7. Menghapus spasi berlebih.
8. Melakukan tokenisasi menggunakan `split()`.
9. Melakukan normalisasi kata tidak baku.
10. Melakukan normalisasi istilah bahasa asing.
11. Menggabungkan kembali token menjadi teks.

Tahapan tersebut bertujuan menghasilkan teks yang lebih bersih dan konsisten.

In [20]:
def preprocessing(text):
    # Pastikan berupa string
    text = str(text)

    # Case folding
    text = text.lower()

    # Hapus URL
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )

    # Hapus email
    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )

    # Hapus angka
    text = re.sub(
        r'\d+',
        ' ',
        text
    )

    # Hapus tanda baca, simbol, dan emotikon
    text = re.sub(
        r'[^a-zA-ZÀ-ÿ\s]',
        ' ',
        text
    )

    # Hapus spasi berlebih
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    # Tokenisasi
    tokens = text.split()

    # Normalisasi kata tidak baku dan bahasa asing
    hasil = []

    for token in tokens:

        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]

        if token in kamus_asing:
            token = kamus_asing[token]

        hasil.append(token)

    # Gabungkan kembali token
    return " ".join(hasil)

## 9. Menerapkan Preprocessing ke Semua Berita

Setelah fungsi preprocessing dibuat, fungsi tersebut diterapkan pada seluruh isi berita.

Hasil preprocessing disimpan pada kolom baru bernama `berita_clean`.

In [22]:
df["berita_clean"] = df["isi_berita"].apply(preprocessing)

df[["isi_berita", "berita_clean"]].head()

,isi_berita,berita_clean
0,"Dalam dua seri terakhir MotoGP 2027, Marc Marq...",dalam dua seri terakhir motogp marc marquez su...
1,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",alwi farhan moh zaki ubaidillah dan muhamad yu...
2,Di tengah viral insiden cepirit pada ajang Hyr...,di tengah viral insiden cepirit pada ajang hyr...
3,Dejan Fedinansyah/Felisha Alberta Nathaniel Pa...,dejan fedinansyah felisha alberta nathaniel pa...
4,Bagas Maulana/Apriyani Rahayu tak minder meski...,bagas maulana apriyani rahayu tak minder meski...


## 10. Menghitung Jumlah Kata Setelah Preprocessing

Setelah teks dibersihkan melalui proses preprocessing, jumlah kata pada setiap berita dihitung kembali.

Perhitungan ini dilakukan untuk melihat perubahan jumlah kata sebelum dan sesudah preprocessing.

Hasil perhitungan disimpan pada kolom baru bernama `jumlah_kata_clean`.

In [23]:
df["jumlah_kata_clean"] = (
    df["berita_clean"]
    .apply(lambda x: len(x.split()))
)

df[
    [
        "id",
        "label",
        "jumlah_kata_asli",
        "jumlah_kata_clean"
    ]
].head()

,id,label,jumlah_kata_asli,jumlah_kata_clean
0,1,sport,320,316
1,2,sport,323,320
2,3,sport,256,261
3,4,sport,238,241
4,5,sport,402,406


## 11. Mengekstrak Seluruh Kata Unik

Setelah seluruh berita melalui proses preprocessing, semua kata dari setiap berita dikumpulkan.

Kemudian kata-kata tersebut diubah menjadi kumpulan kata unik menggunakan `set()`.

Tahap ini dilakukan untuk mengetahui jumlah seluruh kata dan jumlah kata unik yang terdapat pada dataset setelah preprocessing.

In [24]:
semua_kata = []

for berita in df["berita_clean"]:
    semua_kata.extend(berita.split())

kata_unik = sorted(set(semua_kata))

print("Jumlah seluruh kata :", len(semua_kata))
print("Jumlah kata unik :", len(kata_unik))

print("\nContoh kata unik:")
print(kata_unik[:50])

Jumlah seluruh kata : 65434
Jumlah kata unik : 7291

Contoh kata unik:
['a', 'aaa', 'aau', 'ab', 'abad', 'abadi', 'abal', 'abdullah', 'abdurrahkman', 'abha', 'abraham', 'abrahham', 'absen', 'absorber', 'abu', 'academy', 'acara', 'acaranya', 'accelerating', 'acceptance', 'access', 'accessories', 'accident', 'accreditation', 'acd', 'ace', 'aceh', 'achadie', 'achilles', 'achmad', 'acosta', 'acuan', 'ada', 'adalah', 'adam', 'adanya', 'adaptasi', 'adaptif', 'adapun', 'adb', 'adcp', 'ade', 'adelaide', 'adelguer', 'adella', 'adhang', 'adhi', 'adhipramana', 'adhitama', 'adi']


## 12. Membagi Data Training dan Testing

Dataset dibagi menjadi data training dan data testing.

Pembagian data yang digunakan adalah 160 data untuk training dan 40 data untuk testing.

`random_state=42` digunakan agar pembagian data dapat menghasilkan pembagian yang sama ketika kode dijalankan kembali.

Parameter `stratify` digunakan agar jumlah masing-masing label tetap seimbang pada data training dan testing.

In [25]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["berita_clean"],
    df["label_num"],
    test_size=40,
    random_state=42,
    stratify=df["label_num"]
)

print("Jumlah training :", len(X_train_text))
print("Jumlah testing :", len(X_test_text))

print("\nTRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

Jumlah training : 160
Jumlah testing : 40

TRAINING
label_num
0    80
1    80
Name: count, dtype: int64

TESTING
label_num
0    20
1    20
Name: count, dtype: int64


## 13. Representasi TF-IDF

TF-IDF digunakan untuk mengubah data teks menjadi bentuk numerik.

Pada tahap ini, proses `fit_transform()` dilakukan pada data training untuk membentuk vocabulary dan menghitung nilai TF-IDF.

Data testing kemudian menggunakan `transform()` berdasarkan vocabulary yang telah diperoleh dari data training.

Parameter yang digunakan adalah `min_df=2` dan `max_df=0.95`.

In [26]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

nama_fitur = tfidf.get_feature_names_out()

print("Jumlah fitur TF-IDF :", len(nama_fitur))
print("Shape training :", X_train_tfidf.shape)
print("Shape testing :", X_test_tfidf.shape)

Jumlah fitur TF-IDF : 3140
Shape training : (160, 3140)
Shape testing : (40, 3140)


## 14. Melihat Matriks TF-IDF

Hasil proses TF-IDF berupa matriks yang berisi bobot setiap kata pada setiap berita.

Setiap baris mewakili satu berita, sedangkan setiap kolom mewakili satu fitur atau kata.

Matriks training kemudian ditampilkan dalam bentuk DataFrame agar hasil pembobotan dapat dilihat dengan lebih mudah.

In [27]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_train_df.head()

,aaa,abad,abadi,abraham,absen,acara,acd,aceh,achmad,acosta,...,yamaha,yamanaka,yna,yogyakarta,yudha,yudhi,yusuf,zaki,zarco,zona
0,0.0,0.0,0.0,0.0,0.0,0.03531,0.06645,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.030984,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.157699,...,0.0,0.0,0.057111,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


## 15. Menyimpan Data TF-IDF

Hasil TF-IDF untuk data training dan testing disimpan dalam format Excel.

Kolom `label` ditambahkan sebagai kolom terakhir sehingga data hasil TF-IDF tetap mempunyai informasi mengenai kategori berita.

In [28]:
tfidf_train_df["label"] = y_train.reset_index(drop=True)

tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["label"] = y_test.reset_index(drop=True)

tfidf_train_df.to_excel(
    "data_tfidf_training.xlsx",
    index=False
)

tfidf_test_df.to_excel(
    "data_tfidf_testing.xlsx",
    index=False
)

print("Data TF-IDF berhasil disimpan.")

Data TF-IDF berhasil disimpan.


## 16. Perhitungan ICSDF

Setelah mendapatkan nilai TF-IDF, tahap berikutnya adalah menghitung nilai ICSDF.

ICSDF dihitung berdasarkan penyebaran suatu kata pada masing-masing kelas.

Pada dataset ini terdapat dua kelas, yaitu Sport dan Finance.

Untuk setiap kelas dihitung jumlah dokumen yang mengandung suatu kata. Nilai tersebut digunakan untuk mendapatkan kepadatan kata pada masing-masing kelas dan kemudian dihitung nilai ICSDF.

In [29]:
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

for kelas in sorted(y_train.unique()):

    mask = (
        y_train_array == kelas
    )

    X_class = X_train_tfidf[
        mask
    ]

    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )

    jumlah_dokumen_class = (
        X_class.shape[0]
    )

    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )

    class_space_density += (
        class_density
    )

epsilon = 1e-12

icsdf = np.log(
    (jumlah_kelas + epsilon)
    /
    (class_space_density + epsilon)
)

df_icsdf = pd.DataFrame({
    "kata": features,
    "ICSDF": icsdf
})

df_icsdf.sort_values(
    "ICSDF",
    ascending=False
).head(20)

,kata,ICSDF
3119,wietrzyk,4.382027
3116,whole,4.382027
3115,west,4.382027
3114,wellfarm,4.382027
3111,warna,4.382027
3109,wardani,4.382027
30,agreement,4.382027
29,agraria,4.382027
24,adrianto,4.382027
23,adrian,4.382027


## 17. TF-IDF × ICSDF

Setelah nilai ICSDF diperoleh, nilai tersebut digunakan untuk memberikan bobot tambahan pada matriks TF-IDF.

Proses dilakukan dengan mengalikan matriks TF-IDF dengan nilai ICSDF untuk data training dan data testing.

Hasilnya merupakan matriks TF-IDF yang sudah diberi pembobotan berdasarkan ICSDF.

In [30]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

X_train_icsdf = csr_matrix(X_train_icsdf)
X_test_icsdf = csr_matrix(X_test_icsdf)

print("Shape training :", X_train_icsdf.shape)
print("Shape testing :", X_test_icsdf.shape)

Shape training : (160, 3140)
Shape testing : (40, 3140)


## 18. Menentukan Kata Paling Penting

Setelah mendapatkan matriks TF-IDF × ICSDF, setiap fitur diberikan skor berdasarkan nilai rata-ratanya pada data training.

Fitur kemudian diurutkan berdasarkan skor tersebut.

Fitur dengan skor tertinggi dianggap sebagai kata yang paling penting dan akan digunakan pada tahap reduksi fitur berikutnya.

In [31]:
skor_fitur = np.asarray(
    X_train_icsdf.mean(axis=0)
).ravel()

TOP_K = 100

top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

kata_penting = features[top_index]

print("Jumlah fitur terpilih :", len(kata_penting))

print("\nKata paling penting:")
print(kata_penting)

Jumlah fitur terpilih : 100

Kata paling penting:
['marquez' 'purbaya' 'emas' 'pertandingan' 'marc' 'beras' 'motogp' 'poin'
 'lrt' 'asian' 'fortifikasi' 'suahasil' 'bezzecchi' 'martin' 'artikel'
 'djarum' 'aku' 'atlet' 'balapan' 'peserta' 'keuangan' 'rp' 'medali'
 'pejabat' 'manggarai' 'nathan' 'fiskal' 'saham' 'jepang' 'posisi' 'harga'
 'pb' 'moto' 'perombakan' 'prabowo' 'kebijakan' 'hyrox' 'olahraga'
 'indonesia' 'lap' 'saya' 'pramono' 'veda' 'wib' 'stasiun' 'san' 'jakarta'
 'ihsg' 'buruh' 'antam' 'marino' 'sesi' 'kualifikasi' 'konsumen' 'motor'
 'pelajar' 'investasi' 'imam' 'anak' 'detik' 'layanan' 'jorge' 'dki'
 'alex' 'barat' 'menteri' 'pertamina' 'pembalap' 'nagoya' 'bonus' 'ganda'
 'para' 'set' 'dia' 'tni' 'pemerintah' 'juara' 'tim' 'riders' 'bea'
 'pelatih' 'energi' 'kami' 'bandara' 'ojk' 'daerah' 'marco' 'ekonomi'
 'seri' 'ubed' 'pelanggan' 'utang' 'kuarter' 'chandra' 'presiden' 'lari'
 'fadia' 'cukai' 'rkab' 'kita']


## 19. Reduksi Menjadi 100 Fitur ICSDF

Dari seluruh fitur yang tersedia, dipilih 100 fitur dengan skor tertinggi.

Matriks training dan testing kemudian direduksi menggunakan indeks fitur tersebut.

Dengan demikian jumlah fitur berkurang menjadi 100 fitur.

In [32]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

print("Sebelum seleksi :", X_train_tfidf.shape)
print("Sesudah ICSDF :", X_train_selected.shape)

Sebelum seleksi : (160, 3140)
Sesudah ICSDF : (160, 100)


## 20. Tabel Hasil ICSDF

Hasil reduksi 100 fitur kemudian diubah menjadi DataFrame.

Nama kolom menggunakan 100 kata yang telah dipilih sebagai fitur penting.

Label data training ditambahkan sebagai kolom terakhir.

In [33]:
icsdf_train_df = pd.DataFrame(
    X_train_selected.toarray(),
    columns=kata_penting
)

icsdf_train_df["label"] = (
    y_train.reset_index(drop=True)
)

icsdf_train_df.head()

,marquez,purbaya,emas,pertandingan,marc,beras,motogp,poin,lrt,asian,...,utang,kuarter,chandra,presiden,lari,fadia,cukai,rkab,kita,label
0,0.000000,0.508939,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,...,1.201229,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.077630,0
1,0.500480,0.000000,0.000000,0.0,0.433286,0.0,0.149583,0.224374,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1
2,0.000000,0.000000,0.054586,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016046,0
3,0.716891,0.000000,0.000000,0.0,0.223431,0.0,0.321395,0.321395,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.047410,1
4,0.241737,0.000000,0.000000,0.0,0.313922,0.0,0.650250,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1


In [34]:
icsdf_train_df.to_excel(
    "data_icsdf_training.xlsx",
    index=False
)

print("Data ICSDF berhasil disimpan.")

Data ICSDF berhasil disimpan.


## 21. PCA

Setelah fitur direduksi menjadi 100 fitur menggunakan ICSDF, tahap selanjutnya adalah melakukan reduksi dimensi menggunakan Principal Component Analysis (PCA).

Pada tugas ini digunakan 20 komponen utama.

Dengan PCA, 100 fitur hasil ICSDF diubah menjadi 20 fitur baru yang disebut PC1 sampai PC20.

In [35]:
pca = PCA(
    n_components=20,
    random_state=42
)

X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

print("Shape training PCA :", X_train_pca.shape)
print("Shape testing PCA :", X_test_pca.shape)

Shape training PCA : (160, 20)
Shape testing PCA : (40, 20)


## 22. Membuat Tabel Data Reduksi Training

Hasil PCA pada data training kemudian dibuat menjadi DataFrame.

Data hasil PCA terdiri dari 20 komponen utama, yaitu PC1 sampai PC20.

Label ditambahkan sebagai kolom terakhir.

In [36]:
nama_pc = [
    f"PC{i}"
    for i in range(1, 21)
]

df_train_reduksi = pd.DataFrame(
    X_train_pca,
    columns=nama_pc
)

df_train_reduksi["label"] = (
    y_train.reset_index(drop=True)
)

df_train_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.093841,-0.186294,-0.281947,0.158984,0.005162,-0.041209,-0.008122,0.061288,0.006840,-0.138351,...,-0.222510,-0.141928,0.117049,0.080804,0.003380,0.001276,0.082093,-0.016896,-0.023501,0
1,-0.498539,0.324917,0.017074,0.060755,-0.025552,0.040492,0.049642,-0.037678,-0.012044,0.109007,...,0.092555,-0.238864,0.022085,-0.006115,-0.044473,-0.011637,-0.006678,0.010883,-0.021644,1
2,0.031651,-0.049356,-0.010152,-0.039458,0.028821,-0.024913,0.013962,0.045844,0.000539,-0.062344,...,-0.019632,0.001768,-0.048585,-0.058149,-0.013132,-0.008801,-0.024202,0.004673,-0.007526,0
3,-0.577078,0.382586,0.018815,0.078704,-0.026035,0.053313,0.045537,-0.035399,-0.006867,0.114524,...,0.108646,-0.233576,0.021606,0.002834,-0.006552,-0.002335,-0.002952,0.015174,-0.059065,1
4,-0.797316,0.581883,0.040532,0.132478,-0.069127,0.190569,-0.038183,-0.083725,-0.207657,-0.120327,...,-0.236559,0.706538,0.156975,0.139347,0.039791,0.062570,0.075670,-0.034957,0.097710,1


## 23. Membuat Data Testing

Data testing yang sebelumnya telah melalui proses TF-IDF dan ICSDF juga direduksi menggunakan PCA.

Data testing menggunakan transformasi PCA yang telah dibentuk berdasarkan data training.

Hasilnya terdiri dari 20 komponen utama dan satu kolom label.

In [37]:
df_test_reduksi = pd.DataFrame(
    X_test_pca,
    columns=nama_pc
)

df_test_reduksi["label"] = (
    y_test.reset_index(drop=True)
)

df_test_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.246007,-0.501233,1.243587,1.157439,0.002507,0.086201,0.019435,0.035835,-0.028555,0.147322,...,0.188832,0.062754,0.040155,0.093738,-0.067635,0.096219,0.056456,-0.041014,-0.031049,0
1,-0.928065,0.668574,0.038416,0.188229,-0.077654,0.237249,-0.028462,-0.064195,-0.058189,0.172801,...,0.121305,-0.367834,0.087259,0.053660,-0.260079,-0.022690,-0.005266,0.001520,0.130612,1
2,-0.161308,0.080041,0.004882,-0.026020,0.008825,-0.060235,-0.052568,0.071754,-0.062011,-0.041570,...,-0.095716,0.172123,-0.162210,-0.304560,0.633397,0.448599,0.102176,-0.082156,0.043007,1
3,0.053075,-0.122679,0.029632,-0.225322,-0.074650,-0.072086,-0.022795,-0.125903,0.014996,-0.172212,...,0.222116,0.014028,-0.114412,0.083753,-0.011694,-0.023352,0.043826,0.018946,-0.084661,1
4,0.142883,-0.268929,0.409647,0.439829,0.016967,0.001042,0.002120,0.040637,-0.022443,-0.039939,...,-0.100473,-0.079603,0.077702,0.070834,0.031281,-0.005777,-0.016933,0.007078,0.034366,0


## 24. Menyimpan Data Reduksi

Data hasil reduksi PCA untuk training dan testing disimpan dalam format Excel.

File tersebut berisi 20 komponen PCA dan satu kolom label.

In [38]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

print("Data reduksi berhasil disimpan.")

Data reduksi berhasil disimpan.


## 25. Cek Total Data Training dan Testing

Tahap terakhir dilakukan untuk memastikan jumlah data training dan testing sudah sesuai dengan pembagian sebelumnya.

Data training harus memiliki 160 data, sedangkan data testing harus memiliki 40 data.

Selain itu, jumlah kolom harus 21, yaitu 20 komponen PCA dan satu kolom label.

In [39]:
print("Shape training :", df_train_reduksi.shape)
print("Shape testing :", df_test_reduksi.shape)

print("\nJumlah training :", len(df_train_reduksi))
print("Jumlah testing :", len(df_test_reduksi))

Shape training : (160, 21)
Shape testing : (40, 21)

Jumlah training : 160
Jumlah testing : 40


## 26. Memastikan Kolom Terakhir adalah Label

Pemeriksaan terakhir dilakukan untuk memastikan kolom `label` berada pada posisi paling akhir.

Dengan demikian data hasil reduksi memiliki 20 fitur PCA dan satu label sebagai kolom terakhir.

In [40]:
print("Kolom terakhir training :")
print(df_train_reduksi.columns[-1])

print("\nKolom terakhir testing :")
print(df_test_reduksi.columns[-1])

Kolom terakhir training :
label

Kolom terakhir testing :
label
